# 🥊 Fighting Game — Pose Classifier

เก็บท่าทางจากเว็บแคมด้วย **MediaPipe Pose Landmarker** แล้วเทรนโมเดล **RandomForest**
ให้จำแนก 4 ท่า: `neutral`, `punch`, `guard_up`, `block_cross`

ลำดับการรัน:
1. ติดตั้งไลบรารี
2. ดาวน์โหลดโมเดล pose landmarker
3. ฟังก์ชันช่วยเหลือที่ใช้ร่วมกัน (skeleton, smoothing, normalize)
4. ทดสอบตรวจจับโครงกระดูกแบบเรียลไทม์ (ยังไม่จำแนกท่า)
5. เก็บชุดข้อมูลท่าทาง → `pose_dataset.csv`
6. เทรนโมเดล → `pose_classifier.pkl`
7. ทดสอบโมเดลแบบเรียลไทม์


## 1. ติดตั้งไลบรารี

In [4]:
%pip install -q mediapipe opencv-python joblib scikit-learn pandas

^C
Note: you may need to restart the kernel to use updated packages.


## 2. ดาวน์โหลดโมเดล MediaPipe Pose Landmarker

In [ ]:
import urllib.request

MODEL_PATH = "pose_landmarker.task"

urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
    MODEL_PATH,
)
print("บันทึกโมเดลแล้วที่", MODEL_PATH)

## 3. ฟังก์ชันช่วยเหลือที่ใช้ร่วมกัน

ทุกเซลล์ด้านล่างใช้ชุดฟังก์ชันเดียวกันนี้ ทั้งตอนดูสเกเลตันเฉยๆ ตอนเก็บข้อมูล และตอนทำนายจริง
เพื่อให้ฟีเจอร์ที่ป้อนเข้าโมเดล **สอดคล้องกันทุกขั้นตอน**


In [6]:
! pip install -q mediapipe opencv-python joblib scikit-learn pandas


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\ROG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\ROG\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time
import os
MODEL_PATH = "pose_landmarker.task"

# เส้นเชื่อมโครงกระดูก (คู่ index ตาม pose landmark 33 จุดของ MediaPipe)
POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8), (9, 10),
    (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
]

# ยิ่งค่า SMOOTHING_ALPHA น้อย ยิ่งนุ่ม/นิ่งขึ้น แต่หน่วง (delay) มากขึ้นตาม
# ยิ่งค่ามาก ยิ่งตอบสนองไว แต่จุดจะสั่นมากขึ้น ลองปรับ 0.3-0.6 ดู
SMOOTHING_ALPHA = 0.4


def create_pose_landmarker(num_poses=1):
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=num_poses,
    )
    return vision.PoseLandmarker.create_from_options(options)


def open_camera(index=0):
    """เปิดกล้องบน Windows ให้ชัวร์ขึ้น: cv2.VideoCapture(0) เฉยๆ มักเปิดไม่ติด
    เพราะ backend เริ่มต้น (MSMF) มีปัญหา เลยลอง CAP_DSHOW ก่อน แล้วค่อย fallback"""
    for backend in (cv2.CAP_DSHOW, cv2.CAP_ANY):
        cap = cv2.VideoCapture(index, backend)
        if cap.isOpened():
            return cap
        cap.release()

    raise RuntimeError(
        "เปิดกล้องไม่ได้ (index={}). เช็คว่า: 1) ไม่มีแอปอื่นใช้กล้องอยู่ (Zoom/Teams/เบราว์เซอร์แท็บอื่น) "
        "2) Windows Settings > Privacy & security > Camera เปิดสิทธิ์ให้ desktop app แล้ว "
        "3) ลองเปลี่ยน index เป็น 1 ถ้ามีกล้องหลายตัว".format(index)
    )


def landmarks_to_array(landmarks):
    return np.array([[lm.x, lm.y, lm.z] for lm in landmarks])


import time


class OneEuroFilter:
    """สมูท landmark array (33x3) ต่างจาก EMA ตรงที่ปรับความแรง smooth ตามความเร็วการเคลื่อนไหว
    อัตโนมัติ: นิ่งตอนหยุด (ลด jitter) ไวขึ้นตอนขยับเร็ว (ลด lag)
    อ้างอิง: Casiez et al., 2012"""

    def __init__(self, min_cutoff=1.0, beta=0.3, d_cutoff=1.0):
        self.min_cutoff = min_cutoff
        self.beta = beta
        self.d_cutoff = d_cutoff
        self.x_prev = None
        self.dx_prev = None
        self.t_prev = None

    @staticmethod
    def _alpha(cutoff, dt):
        tau = 1.0 / (2 * np.pi * cutoff)
        return 1.0 / (1.0 + tau / dt)

    def __call__(self, x, t=None):
        x = np.asarray(x, dtype=np.float64)
        t = time.time() if t is None else t

        if self.x_prev is None:
            self.x_prev = x
            self.dx_prev = np.zeros_like(x)
            self.t_prev = t
            return x

        dt = max(t - self.t_prev, 1e-6)

        dx = (x - self.x_prev) / dt
        a_d = self._alpha(self.d_cutoff, dt)
        dx_hat = a_d * dx + (1 - a_d) * self.dx_prev

        # ขยับเร็ว -> cutoff สูง -> smooth น้อยลง (ตอบสนองไว)
        cutoff = self.min_cutoff + self.beta * np.abs(dx_hat)
        a_x = self._alpha(cutoff, dt)
        x_hat = a_x * x + (1 - a_x) * self.x_prev

        self.x_prev, self.dx_prev, self.t_prev = x_hat, dx_hat, t
        return x_hat

    def reset(self):
        self.x_prev = self.dx_prev = self.t_prev = None



def normalize_points(pts):
    """ทำให้ landmark ไม่ขึ้นกับตำแหน่ง/ระยะห่างจากกล้อง
    อิงจุดกึ่งกลางสะโพกเป็น origin และ scale ด้วยระยะไหล่-สะโพก"""
    hip_center = (pts[23] + pts[24]) / 2
    shoulder_center = (pts[11] + pts[12]) / 2
    scale = np.linalg.norm(shoulder_center - hip_center)
    if scale < 1e-6:
        scale = 1.0
    normalized = (pts - hip_center) / scale
    return normalized.flatten()  # 33*3 = 99 ค่า


def draw_skeleton(frame, pts, w, h):
    for x, y, _ in pts:
        cv2.circle(frame, (int(x * w), int(y * h)), 4, (0, 255, 0), -1)
    for a, b in POSE_CONNECTIONS:
        xa, ya, _ = pts[a]
        xb, yb, _ = pts[b]
        cv2.line(frame, (int(xa * w), int(ya * h)), (int(xb * w), int(yb * h)), (255, 255, 255), 2)

## 4. ทดสอบตรวจจับโครงกระดูกแบบเรียลไทม์ (ยังไม่จำแนกท่า)

In [3]:
def open_camera(index=0, warmup_frames=8):
    """เปิดกล้องบน Windows ให้ชัวร์ขึ้น และกันปัญหาภาพดำ
    (มักเกิดจาก backend/pixel-format ไม่เข้ากับกล้อง หรือกล้องยังไม่ warm up)"""
    backends = [cv2.CAP_DSHOW, cv2.CAP_MSMF, cv2.CAP_ANY]

    for idx in (index, 1 - index):  # ลอง index ที่ขอก่อน แล้วลอง index อื่น (บางเครื่องมีกล้อง IR แยกจาก RGB)
        for backend in backends:
            cap = cv2.VideoCapture(idx, backend)
            if not cap.isOpened():
                cap.release()
                continue

            cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))

            for _ in range(warmup_frames):
                success, frame = cap.read()
                if success and frame is not None and frame.mean() > 5:  # ไม่ใช่จอดำสนิท
                    print(f"เปิดกล้อง index={idx} backend={backend} สำเร็จ")
                    return cap

            cap.release()

    raise RuntimeError(
        "เปิดกล้องไม่ได้ หรือได้แต่ภาพดำ เช็คว่า: "
        "1) ไม่มีแอปอื่นใช้กล้องอยู่ (Zoom/Teams/เบราว์เซอร์แท็บอื่น) "
        "2) Windows Settings > Privacy & security > Camera เปิดสิทธิ์ desktop app แล้ว "
        "3) ถ้าเครื่องมีกล้อง IR (Windows Hello) โค้ดนี้จะลอง index 1 ให้อัตโนมัติ แต่ถ้ายังไม่ได้ลองแก้ index เองเพิ่ม"
    )

    cap.release()
    cv2.destroyAllWindows()




## 5. เก็บชุดข้อมูลท่าทาง

ท่าที่เก็บ: `neutral` (ยืนเฉย), `punch` (ยื่นชก), `guard_up` (ยกมือ 2 ข้างรอ counter), `block_cross` (กากบาทป้องกัน)

**วิธีใช้**
- กดค้าง `n` / `p` / `g` / `b` เพื่อบันทึกท่านั้นๆ (ขยับท่าเล็กน้อยระหว่างกดค้าง เพื่อให้ข้อมูลหลากหลาย)
- แนะนำเก็บอย่างน้อย 150-300 ตัวอย่างต่อคลาส
- กด `q` เพื่อออกและบันทึกไฟล์ `pose_dataset.csv`


In [7]:
import csv
import os

CSV_PATH = "pose_dataset4.csv"

LABELS = {
    ord("n"): "neutral",
    ord("p"): "punch",
    ord("g"): "guard_up",
    ord("b"): "block_cross",
}

WINDOW_NAME = "Data Collector (n/p/g/b to hold, or click buttons to toggle, q to quit)"
BUTTON_ORDER = ["neutral", "punch", "guard_up", "block_cross"]
BUTTON_W, BUTTON_H, BUTTON_MARGIN = 150, 40, 10


def get_button_rects():
    rects = {}
    x = BUTTON_MARGIN
    y = 100  # เว้นที่ด้านบนไว้ให้ตัวนับ count เดิม
    for name in BUTTON_ORDER:
        rects[name] = (x, y, x + BUTTON_W, y + BUTTON_H)
        y += BUTTON_H + BUTTON_MARGIN
    return rects


def draw_buttons(frame, rects, active_label):
    for name, (x1, y1, x2, y2) in rects.items():
        color = (0, 200, 0) if name == active_label else (90, 90, 90)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, -1)
        cv2.putText(frame, name, (x1 + 8, y1 + 27), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)


def make_mouse_callback(rects, state):
    def on_mouse(event, x, y, flags, param):
        if event != cv2.EVENT_LBUTTONDOWN:
            return
        for name, (x1, y1, x2, y2) in rects.items():
            if x1 <= x <= x2 and y1 <= y <= y2:
                # คลิกปุ่มที่กำลัง active อยู่ = หยุด, คลิกปุ่มอื่น = สลับไปบันทึกท่านั้นแทน
                state["active"] = None if state["active"] == name else name
                break
    return on_mouse


def collect_dataset():
    detector = create_pose_landmarker()

    file_exists = os.path.exists(CSV_PATH)
    csv_file = open(CSV_PATH, "a", newline="")
    writer = csv.writer(csv_file)
    if not file_exists:
        header = ["label"] + [f"{axis}{i}" for i in range(33) for axis in ("x", "y", "z")]
        writer.writerow(header)

    counts = {name: 0 for name in LABELS.values()}
    cap = open_camera()
    frame_ts = 0
    smoothed_pts = None
    pose_filter = OneEuroFilter(min_cutoff=1.0, beta=0.3)

    button_rects = get_button_rects()
    mouse_state = {"active": None}

    cv2.namedWindow(WINDOW_NAME)
    cv2.setMouseCallback(WINDOW_NAME, make_mouse_callback(button_rects, mouse_state))

    print("กด n/p/g/b ค้างไว้ = บันทึกท่าเดียวมือเดียวได้")
    print("หรือคลิกปุ่มที่มุมซ้ายบน = toggle บันทึกต่อเนื่อง ปล่อยมือทำท่าสองมือได้ คลิกซ้ำเพื่อหยุด")
    print("กด q (คลิกโฟกัสหน้าต่างก่อน) เพื่อออก")

    try:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = detector.detect_for_video(mp_image, frame_ts)
            frame_ts += 33

            h, w, _ = frame.shape
            key = cv2.waitKey(1) & 0xFF

            record_label = LABELS.get(key) or mouse_state["active"]

            if result.pose_landmarks:
                raw_pts = landmarks_to_array(result.pose_landmarks[0])
                smoothed_pts = pose_filter(raw_pts, t=time.time())
                draw_skeleton(frame, smoothed_pts, w, h)

                if record_label:
                    features = normalize_points(smoothed_pts)
                    writer.writerow([record_label] + features.tolist())
                    csv_file.flush()
                    counts[record_label] += 1
            else:
                smoothed_pts = None
                pose_filter.reset()

            display = cv2.flip(frame, 1)

            y0 = 30
            for name, c in counts.items():
                cv2.putText(display, f"{name}: {c}", (10, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
                y0 += 25

            draw_buttons(display, button_rects, mouse_state["active"])
            if mouse_state["active"]:
                cv2.putText(display, f"REC: {mouse_state['active']}", (10, h - 20),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            cv2.imshow(WINDOW_NAME, display)
            if key == ord("q"):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        csv_file.close()
        print("บันทึกเสร็จแล้วที่", CSV_PATH)
        print("จำนวนตัวอย่างต่อคลาส:", counts)


collect_dataset()

เปิดกล้อง index=0 backend=700 สำเร็จ
กด n/p/g/b ค้างไว้ = บันทึกท่าเดียวมือเดียวได้
หรือคลิกปุ่มที่มุมซ้ายบน = toggle บันทึกต่อเนื่อง ปล่อยมือทำท่าสองมือได้ คลิกซ้ำเพื่อหยุด
กด q (คลิกโฟกัสหน้าต่างก่อน) เพื่อออก
บันทึกเสร็จแล้วที่ pose_dataset4.csv
จำนวนตัวอย่างต่อคลาส: {'neutral': 158, 'punch': 205, 'guard_up': 155, 'block_cross': 167}


## 6. เทรนโมเดลจำแนกท่าทาง (RandomForest)

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

CSV_PATH = "pose_dataset4.csv"
MODEL_OUT = "pose_classifier2.pkl"

df = pd.read_csv(CSV_PATH)

train_parts, test_parts = [], []
for label, group in df.groupby("label"):
    split_idx = int(len(group) * 0.8)
    train_parts.append(group.iloc[:split_idx])
    test_parts.append(group.iloc[split_idx:])

train_df = pd.concat(train_parts)
test_df = pd.concat(test_parts)

X_train, y_train = train_df.drop("label", axis=1), train_df["label"]
X_test, y_test = test_df.drop("label", axis=1), test_df["label"]

clf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

joblib.dump(clf, MODEL_OUT)
print("บันทึกโมเดลแล้วที่", MODEL_OUT)

              precision    recall  f1-score   support

 block_cross       0.93      0.49      0.64       160
    guard_up       0.65      0.99      0.78       137
     neutral       0.80      0.62      0.70       136
       punch       0.71      0.89      0.79       141

    accuracy                           0.74       574
   macro avg       0.77      0.75      0.73       574
weighted avg       0.78      0.74      0.73       574

บันทึกโมเดลแล้วที่ pose_classifier2.pkl


## 7. ทดสอบโมเดลแบบเรียลไทม์

In [ ]:
def live_predict():
    clf = joblib.load("pose_classifier.pkl")
    detector = create_pose_landmarker()

    cap = open_camera()
    frame_ts = 0
    smoothed_pts = None
    pose_filter = OneEuroFilter(min_cutoff=1.0, beta=0.3)

    print("กด ESC (ต้องคลิกโฟกัสหน้าต่างวิดีโอก่อน) เพื่อออก — ห้ามกด Interrupt ใน Jupyter แทน")

    try:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = detector.detect_for_video(mp_image, frame_ts)
            frame_ts += 33

            h, w, _ = frame.shape
            label_text = "no pose"

            if result.pose_landmarks:
                raw_pts = landmarks_to_array(result.pose_landmarks[0])
                smoothed_pts = pose_filter(raw_pts, t=time.time())
                draw_skeleton(frame, smoothed_pts, w, h)

                features = normalize_points(smoothed_pts).reshape(1, -1)
                pred = clf.predict(features)[0]
                confidence = clf.predict_proba(features).max()
                label_text = f"{pred} ({confidence * 100:.0f}%)"
            else:
                smoothed_pts = None
                pose_filter.reset()

            display = cv2.flip(frame, 1)
            cv2.putText(display, label_text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
            cv2.imshow("Live Pose Prediction (ESC to quit)", display)
            if cv2.waitKey(5) & 0xFF == 27:
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()


live_predict()

เปิดกล้อง index=0 backend=700 สำเร็จ
กด ESC (ต้องคลิกโฟกัสหน้าต่างวิดีโอก่อน) เพื่อออก — ห้ามกด Interrupt ใน Jupyter แทน


C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Py

In [21]:
def live_predict():
    clf = joblib.load("pose_classifier.pkl")
    detector = create_pose_landmarker()

    cap = open_camera()
    frame_ts = 0
    smoothed_pts = None
    pose_filter = OneEuroFilter(min_cutoff=1.0, beta=0.3)

    print("กด ESC (ต้องคลิกโฟกัสหน้าต่างวิดีโอก่อน) เพื่อออก — ห้ามกด Interrupt ใน Jupyter แทน")

    try:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = detector.detect_for_video(mp_image, frame_ts)
            frame_ts += 33

            h, w, _ = frame.shape
            label_text = "no pose"

            if result.pose_landmarks:
                raw_pts = landmarks_to_array(result.pose_landmarks[0])
                smoothed_pts = pose_filter(raw_pts, t=time.time())
                draw_skeleton(frame, smoothed_pts, w, h)

                features = normalize_points(smoothed_pts).reshape(1, -1)
                pred = clf.predict(features)[0]
                confidence = clf.predict_proba(features).max()
                label_text = f"{pred} ({confidence * 100:.0f}%)"
            else:
                smoothed_pts = None
                pose_filter.reset()

            display = cv2.flip(frame, 1)
            cv2.putText(display, label_text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
            cv2.imshow("Live Pose Prediction (ESC to quit)", display)
            if cv2.waitKey(5) & 0xFF == 27:
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()


live_predict()

RuntimeError: เปิดกล้องไม่ได้ หรือได้แต่ภาพดำ เช็คว่า: 1) ไม่มีแอปอื่นใช้กล้องอยู่ (Zoom/Teams/เบราว์เซอร์แท็บอื่น) 2) Windows Settings > Privacy & security > Camera เปิดสิทธิ์ desktop app แล้ว 3) ถ้าเครื่องมีกล้อง IR (Windows Hello) โค้ดนี้จะลอง index 1 ให้อัตโนมัติ แต่ถ้ายังไม่ได้ลองแก้ index เองเพิ่ม

In [9]:
df = pd.read_csv(CSV_PATH)
print(df["label"].value_counts())

label
neutral        961
block_cross    960
guard_up       887
punch          780
Name: count, dtype: int64
